In [1]:
import os
from getpass import getpass
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool

# 1. API Key Setup
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

# 2. Tool Definition
@tool
def search_internal_knowledge(query: str) -> str:
    """Useful for answering queries from internal company documents, Q3 reports, or knowledge base."""
    return f"Internal Knowledge Base Results for '{query}': The company Q3 revenue grew by 15% driven by advanced AI automation workflows."

tools = [search_internal_knowledge]
tool_node = ToolNode(tools)

# 3. Model Setup & Node Functions
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

model_with_tools = model.bind_tools(tools)

def call_model(state: MessagesState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 4. Workflow Building & Compilation
workflow = StateGraph(MessagesState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

app = workflow.compile()
print("Agent graph compiled successfully!")

# 5. Execution / Testing
query = "What is our Q3 revenue?"
initial_state = {"messages": [HumanMessage(content=query)]}
result = app.invoke(initial_state)

print("\n--- AI Response ---")
print(result["messages"][-1].content)

Enter your Groq API key:  ········


Agent graph compiled successfully!

--- AI Response ---
Our internal knowledge base indicates that **Q3 revenue grew by 15%**, driven largely by our advanced AI automation workflows. However, the exact dollar amount for Q3 revenue isn’t listed in the documents I’ve accessed. If you need the precise figure, I can look it up in the detailed financial statements or ask the finance team for the specific number. Let me know how you’d like to proceed!


In [2]:
import os
import requests

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)
print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'object': 'list', 'data': [{'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512, 'name': 'Llama Prompt Guard 2 22M', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 512, 'max_output_length': 512, 'pricing': {'prompt': '0.00000003', 'completion': '0.00000003', 'image': '0', 'request': '0', 'input_cache_read': '0.000000015'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens']}, {'model_card': 'https://openai.com/index/gpt-oss-model-card', 'id': 'openai/gpt-oss-120b', 'object': 'model', 'created': 1754408224, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536, 'hugging_face_id': 'openai/gpt-oss-120b', 'name': 'GPT OSS 120B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'con